# Lab 08 — A QC screen and the 'what must survive' question

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 8 — §8.2 (quality control before preprocessing), §8.6 (powerline interference), §8.8 (baseline wander), §8.11 (the diagnostic decision framework).

**Biomedical question.** Is this recording fit — and what must a correction NOT damage?
**Task type (§1.8).** Quality control / triage BEFORE analysis (screen the recording, then decide what — if anything — to correct)
**Information that must be preserved.** for a diagnostic ECG, the ST-segment / repolarisation morphology and QRS timing
**Main assumptions.** the three problems are additive and mostly band-separated from the diagnostic features; a local TP baseline makes the ST level directly measurable
**Primary diagnostic.** a QC screen (narrow-50 Hz fraction, sub-0.5 Hz drift fraction, sliding-window burst z) + a MEASURED ST error under a safe notch vs a naive high-pass
**Transfer challenge.** redo the triage for a 60 Hz-mains / paced-rhythm recording, or where the T-wave (not the ST level) is the feature that must survive

*Self-contained: one seeded synthetic ECG (a clean ground-truth + a corrupted copy), `np.random.default_rng(2013)`, `scipy.signal` SOS filtering, no `bsp`, no I/O; runs offline in well under a minute.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab08_qc_screen_what_must_survive/lab08_qc_screen_what_must_survive.ipynb) [![nbviewer](https://img.shields.io/badge/view-nbviewer-orange)](https://nbviewer.org/github/farhad-abtahi/CM2013/blob/main/labs/lab08_qc_screen_what_must_survive/lab08_qc_screen_what_must_survive.ipynb) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab08_qc_screen_what_must_survive.ipynb)

In [ ]:
# --- shared setup (reproducible; fully offline synthetic ECG, no bsp / no I/O) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

fs   = 500                                   # Hz — diagnostic-ECG sampling rate
secs = 30
t    = np.arange(int(fs*secs))/fs
RR   = 0.8                                    # regular sinus rhythm ~75 bpm
R_TIMES  = np.arange(0.6, t[-1]-0.5, RR)      # KNOWN QRS times  (ground-truth timing)
ST_LEVEL = 0.12                              # KNOWN ST-segment level (arb. mV) — the truth a
                                             #   correction must NOT damage
BURST_T0, BURST_T1 = 12.0, 12.4              # a short motion burst lives in this window

def _beat(tc):
    """One PQRST beat centred on R-peak time tc, with a flat ST plateau of height ST_LEVEL."""
    return ( 0.10*np.exp(-((t-(tc-0.160))/0.025)**2)      # P
            -0.10*np.exp(-((t-(tc-0.020))/0.008)**2)       # Q
            +1.00*np.exp(-((t-(tc-0.000))/0.008)**2)       # R
            -0.15*np.exp(-((t-(tc+0.020))/0.010)**2)       # S
            +ST_LEVEL*np.exp(-((t-(tc+0.100))/0.050)**2)   # ST plateau (centred +100 ms, ~flat)
            +0.30*np.exp(-((t-(tc+0.300))/0.040)**2))      # T

# GROUND TRUTH: a clean diagnostic ECG we keep for comparison
clean = sum(_beat(tc) for tc in R_TIMES) + 0.015*rng.standard_normal(t.size)

# CORRUPTED COPY: three additive problems on top of the SAME clean beats
mains = 0.08*np.sin(2*np.pi*50*t)                                        # (1) 50 Hz mains
drift = 0.25*np.sin(2*np.pi*0.30*t + 0.7) + 0.12*np.sin(2*np.pi*0.15*t)  # (2) baseline wander ~0.3 Hz
burst = np.zeros_like(t)
_b0, _b1 = int(BURST_T0*fs), int(BURST_T1*fs)
burst[_b0:_b1] = 0.5*rng.standard_normal(_b1-_b0)                        # (3) broadband motion burst
corrupt = clean + mains + drift + burst

def measure_st(x):
    """Clinical-style ST reader: mean of a +80..+120 ms ST window MINUS a local TP baseline
    (-300..-220 ms) taken just before each beat. The local baseline cancels slow drift to first
    order, so this returns the ST LEVEL, not the wander. Averaged over beats."""
    v = []
    for tc in R_TIMES:
        if tc-0.30 < 0 or tc+0.13 > t[-1]:
            continue
        base = x[int((tc-0.30)*fs):int((tc-0.22)*fs)].mean()
        stv  = x[int((tc+0.08)*fs):int((tc+0.12)*fs)].mean()
        v.append(stv - base)
    return float(np.mean(v))

# The ST level a VALID correction must return (measured on the clean truth; the estimator has a
# tiny intrinsic bias vs the injected ST_LEVEL, so we compare filters against THIS measured truth).
ST_TRUTH = measure_st(clean)
print(f"clean ECG {clean.shape}  fs={fs} Hz  beats={R_TIMES.size}")
print(f"ST level: injected={ST_LEVEL:.3f}   clean-measured truth ST_TRUTH={ST_TRUTH:.4f}")


## 0. See the recording — one clean, one corrupted
The two traces share the SAME beats; the corrupted copy just adds mains, a slow wander, and a motion burst. Everything below is about *screening* those three and deciding what — if anything — is safe to filter.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
sl = slice(int(10.5*fs), int(14.5*fs))                       # a window that contains the 12.0-12.4 s burst
ax[0].plot(t[sl], clean[sl]); ax[0].set_title("clean ECG (ground truth)"); ax[0].set_ylabel("mV")
ax[1].plot(t[sl], corrupt[sl], color="C3"); ax[1].set_ylabel("mV"); ax[1].set_xlabel("s")
ax[1].set_title("corrupted copy (mains + baseline wander + motion burst)")
ax[1].axvspan(BURST_T0, BURST_T1, color="k", alpha=.12)
plt.tight_layout(); plt.show()
# Checkpoint: which of the three problems can you SEE here, and which is easier to HEAR in the spectrum?

## 1. QC screen — detect each problem from the signal (TODO)
`# TODO` complete three detectors. A QC screen is the first thing you run on ANY new recording: it must FIRE on defects and stay quiet on a clean trace.

In [ ]:
def qc_screen(x, fs):
    """Return a QC report: a narrow-50 Hz mains fraction, a low-frequency drift fraction, and a
    motion-burst detector (robust z of a sliding-window variance, plus where it fires)."""
    qc = {}
    f, P = sig.welch(x, fs=fs, nperseg=4096)                 # spectrum for the two spectral tests

    # TODO (1) MAINS: fraction of total power inside a NARROW 49-51 Hz band. A tall thin 50 Hz
    #   spike is the mains signature. Set qc["mains50_frac"].
    raise NotImplementedError("TODO: implement this — see the comment above")

    # TODO (2) DRIFT: fraction of total power BELOW ~0.5 Hz (slow baseline wander lives here).
    #   Set qc["drift_frac"].
    raise NotImplementedError("TODO: implement this — see the comment above")

    # TODO (3) MOTION BURST: slide a 0.2 s window, take its VARIANCE, and flag a short stretch of
    #   anomalously high variance with a ROBUST z-score (median / MAD, so a handful of R-peaks do
    #   not set the scale). Set qc["burst_z"] (peak z) and qc["burst_t_s"] (where it fires).
    raise NotImplementedError("TODO: implement this — see the comment above")
    return qc

qc_corrupt = qc_screen(corrupt, fs)
qc_clean   = qc_screen(clean, fs)                            # control: a clean trace must read ~0
print(f"{'QC metric':<16}{'corrupted':>12}{'clean (control)':>18}")
print(f"{'mains50_frac':<16}{qc_corrupt['mains50_frac']:>12.4f}{qc_clean['mains50_frac']:>18.5f}")
print(f"{'drift_frac':<16}{qc_corrupt['drift_frac']:>12.4f}{qc_clean['drift_frac']:>18.5f}")
print(f"{'burst_z':<16}{qc_corrupt['burst_z']:>12.1f}{qc_clean['burst_z']:>18.1f}"
      f"   (burst at t={qc_corrupt['burst_t_s']:.1f} s)")
# Checkpoint: each corrupted metric sits far above its clean control -> all three problems DETECTED.

## 2. The first decision (Ch 8) — before you filter anything (TODO)
`# TODO` classify each detected pattern as **remove** / **mark** / **it-is-a-source** / **measurement-failed**, and say — in one sentence — why filtering *first* is dangerous. Naming what a pattern IS comes BEFORE any correction.

In [ ]:
# TODO give each pattern a decision (category from: remove / mark / it-is-a-source /
#   measurement-failed) with one reason, and set `why_filter_first_is_dangerous` to one sentence.
raise NotImplementedError("TODO: implement this — see the comment above")
print(f"{'pattern':<26}{'decision':<26}why")
for pat, cat, why in decisions:
    print(f"{pat:<26}{cat:<26}{why}")
print("\nWhy filtering FIRST is dangerous:\n  " + why_filter_first_is_dangerous)

## 3. What must survive — the notch keeps ST, a naive high-pass destroys it (TODO)
`# TODO` (a) remove the mains with a 50 Hz **notch built as second-order sections** and confirm the ST level (and QRS timing) survive; (b) then run a NAIVE aggressive **2 Hz high-pass** that 'cleans' the drift, and MEASURE the ST error it introduces. Filter zero-phase (`sosfiltfilt`).

In [ ]:
# TODO (a) 50 Hz notch as SOS, applied zero-phase; then measure ST and its error vs ST_TRUTH.
raise NotImplementedError("TODO: implement this — see the comment above")

# TODO (b) NAIVE aggressive high-pass at 2 Hz (SOS, zero-phase); measure the ST error it introduces.
raise NotImplementedError("TODO: implement this — see the comment above")

# QRS timing preservation under the notch, measured OUTSIDE the marked (measurement-failed) burst.
def _rpeaks(x):
    pk, _ = sig.find_peaks(x, height=0.5, distance=int(0.4*fs)); return pk/fs
rc, rn = _rpeaks(clean), _rpeaks(notched)
qrs_shift_ms = max(abs(rn[np.argmin(np.abs(rn - tc))] - tc)*1000
                   for tc in rc if not (BURST_T0-0.3 <= tc <= BURST_T1+0.3))

print(f"ST truth (clean)      = {ST_TRUTH:.4f}")
print(f"ST after 50 Hz notch  = {st_notch:.4f}   relative error = {100*err_notch:5.1f}%   (PRESERVED)")
print(f"ST after 2 Hz hi-pass = {st_hp:.4f}   relative error = {100*err_hp:5.1f}%   (DESTROYED)")
print(f"max QRS timing shift under notch (excl. burst) = {qrs_shift_ms:.2f} ms")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
tc0 = 2.2                                                     # one clean beat, far from the burst
sl  = slice(int((tc0-0.35)*fs), int((tc0+0.45)*fs))
for a in ax:
    a.axvspan(tc0+0.08, tc0+0.12, color="C2", alpha=.15)      # the ST measurement window
    a.plot(t[sl], clean[sl], "k", lw=2, label="clean truth"); a.set_xlabel("s")
ax[0].plot(t[sl], notched[sl], "C0", label="50 Hz notch")
ax[0].set_title("notch: ST level + QRS preserved"); ax[0].legend(fontsize=8)
ax[1].plot(t[sl], hp[sl], "C3", label="naive 2 Hz high-pass")
ax[1].set_title("aggressive HP: ST dragged to zero"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
# Checkpoint: on the right the ST plateau is pulled toward zero and the T-wave is warped -- the
# high-pass 'cleaned' the drift but ate the repolarisation morphology a diagnosis depends on.

## 4. Live sanity check
A preservation claim you never test is untrustworthy. Assert — from the LIVE numbers — that the notch keeps the ST level (small error) and QRS timing, while the aggressive high-pass corrupts the ST (large error); and that all three detectors fire only on the corrupted trace.

In [ ]:
# --- sanity check: notch preserves 'what must survive'; aggressive HP breaks it (computed live) ---
assert err_notch < 0.10, f"notch should preserve ST within 10%, got {100*err_notch:.1f}%"
assert qrs_shift_ms < 5.0, f"zero-phase notch should keep QRS timing, got {qrs_shift_ms:.2f} ms"
assert err_hp > 0.40, f"aggressive 2 Hz HP should corrupt ST by >40%, got {100*err_hp:.1f}%"
assert qc_corrupt["mains50_frac"] > 20*qc_clean["mains50_frac"], "mains detector must fire on corrupt only"
assert qc_corrupt["drift_frac"] > 0.10 and qc_clean["drift_frac"] < 0.01, "drift detector must fire on corrupt only"
assert qc_corrupt["burst_z"] > 8 and qc_clean["burst_z"] < 5, "burst detector must fire on corrupt only"
print(f"[ok] mains detected : corrupt {qc_corrupt['mains50_frac']:.4f}  vs clean {qc_clean['mains50_frac']:.5f}")
print(f"[ok] drift detected : corrupt {qc_corrupt['drift_frac']:.4f}  vs clean {qc_clean['drift_frac']:.5f}")
print(f"[ok] burst detected : z={qc_corrupt['burst_z']:.1f} at t={qc_corrupt['burst_t_s']:.1f}s  (clean z={qc_clean['burst_z']:.1f})")
print(f"[ok] notch PRESERVES ST: {100*err_notch:.1f}% error, QRS shift {qrs_shift_ms:.2f} ms")
print(f"[ok] aggressive HP CORRUPTS ST: {100*err_hp:.1f}% error  <- the 'what must survive' rule is broken")

## Reflection
1. **Stable vs fragile QC calls.** Recompute the mains fraction with a 48–52 Hz band and the drift fraction with a 0.3 Hz vs 0.5 Hz cutoff. Which QC conclusion (*mains present? drift present? burst present?*) stays stable across these reasonable thresholds, and which is closest to flipping?
2. **Correct vs mark.** For this *diagnostic* ECG, which problems would you correct before ST analysis and which would you only MARK, and what live evidence (ST error vs the clean truth, the burst z) justifies each choice?
3. **Transfer.** What would you check before trusting this QC screen on a new device or lead (60 Hz mains, a different sampling rate, a higher noise floor, a paced rhythm)?

**Rule out.** A pipeline that *filters before diagnosing* — or any baseline-wander high-pass with a corner high enough to flatten the ST segment (here the naive 2 Hz high-pass, which introduced a ~70% ST error) — is ruled out: it breaks the §1.8 *information-that-must-be-preserved* requirement and the §8.2 *'what must survive'* rule for a diagnostic ECG, namely the **ST-segment / repolarisation morphology and QRS timing**.

> *Your answers here.*